In [1]:
import os
import random
import pandas as pd
import numpy as np
from glob import glob
import time
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

In [2]:
random.seed(0)

# Most important features (top 5 by SHAP):
# doublecheck: Numerator of the MP2 t2-amplitude, two-electron integral <ik || ab>
# t2start: Initial MP2 t2-amplitude
# t2mag: Magnitude of the MP2 t2-amplitude
# orbdiff: Denominator of the MP2 t2-amplitude
# diag: Binary feature denoting whether a=b (virtual orbits are the same)
feat = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']

# Feature order in X
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

# get training molecules
basis_sets = ['STO_3G', 'cc_pVDZ','aug_cc_pVDZ'] 

In [5]:
steps = [20,40,60,80,100]
for basis in basis_sets: 
    for n in steps:
        t1 = time.time()
        # get training molecules
        fn = os.path.join(os.path.expanduser("~"), "DDLUCJ", "DDLUCJ_Models", f"faster_{basis}_data_splits_{n}.pkl")
        with open(fn, "rb") as f:
            data = joblib.load(f)
        
        
        X_train = data["X_train"]
        y_train = data["y_train"]
        X_test = data["X_test"]
        y_test = data["y_test"]
    
        # Instantiate model
        model = XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        tree_method="hist",   # faster + smaller
        n_jobs=-1,
        random_state=42
    )
        
        model_pipeline = Pipeline([
            ('scaler', MinMaxScaler(feature_range=(-1,1))),
            ('regressor', model)
        ])
        
        # fit model
        model_pipeline.fit(X_train, y_train)
        y_pred = model_pipeline.predict(X_test)
        
        # compute performance metrics
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
    
        # save metrics
        print(f"N: {n}, MAE: {mae}, MSE: {mse}, R2: {r2}")
        with open(f"out/xgboost_{basis}_performance.txt", "a") as f:
            f.write(f"N: {n}, MAE: {mae}, MSE: {mse}, R2: {r2}\n")
        
        
        # save model and train test splits
        #joblib.dump(model_pipeline, f"out/xgboost_faster_sto_3g_pVDZ_model_{n}.pkl")
        
        model.save_model(f"out/{basis}_xgb_model_{n}.json")
        t2 = time.time()
        print(f"{n} samples took {t2-t1} seconds\n")


N: 20, MAE: 0.0002330152269207157, MSE: 7.532092223466864e-07, R2: 0.9860810891530944
20 samples took 1.5365025997161865 seconds

N: 40, MAE: 0.0002176263665407508, MSE: 5.681371028847376e-07, R2: 0.989501124721715
40 samples took 2.0125017166137695 seconds

N: 60, MAE: 0.0002014400357812678, MSE: 5.077976929190731e-07, R2: 0.9906161653278965
60 samples took 1.7868943214416504 seconds

N: 80, MAE: 0.00019647668561375374, MSE: 4.776372338980758e-07, R2: 0.9911735147704686
80 samples took 2.1978800296783447 seconds

N: 100, MAE: 0.00019005345457551788, MSE: 3.7697602708236656e-07, R2: 0.9930336810056144
100 samples took 2.5854859352111816 seconds

